# 5686: clinical plausibility audit of Delphi-2M generations

Rule-based screening plus an LLM judge over trajectories sampled from a Delphi demo model trained on the authors' synthetic release. Companion to research proposal 5686.

## 1. Setup

In [1]:
!git clone https://github.com/negin-kafee/Delphi.git
%cd Delphi
!pip install -r requirements.txt
!python train.py config/train_delphi_demo.py --device=cuda --out_dir=Delphi-2M

Cloning into 'Delphi'...
remote: Enumerating objects: 213, done.
remote: Counting objects: 100% (134/134), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 213 (delta 97), reused 58 (delta 58), pack-reused 79 (from 1)
Receiving objects: 100% (213/213), 31.25 MiB | 12.24 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/Delphi
ERROR: Could not find a version that satisfies the requirement torch==2.3.0 (from versions: 2.5.0, 2.5.1, 2.6.0, 2.7.0, 2.7.1, 2.8.0, 2.9.0, 2.9.1, 2.10.0, 2.11.0, 2.12.0, 2.12.1, 2.13.0, 2.14.0)
ERROR: No matching distribution found for torch==2.3.0
Overriding config with config/train_delphi_demo.py:

import time

out_dir = 'Delphi'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 25
log_interval = 25 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 

In [2]:
!pip install -q torch numpy pandas matplotlib scikit-learn umap-learn shap tqdm

In [3]:
import os, glob
print("cwd:", os.getcwd())
print("here:", os.listdir('.')[:20])
print("\nfound labels.csv at:")
print(glob.glob('/content/**/labels.csv', recursive=True))
print("\nfound train.bin at:")
print(glob.glob('/content/**/train.bin', recursive=True))

cwd: /content/Delphi
here: ['requirements.txt', 'supplementary', '.github', 'LICENSE', 'Delphi-2M', 'delphi_labels_chapters_colours_icd.csv', 'sampling_trajectories.ipynb', 'plotting.py', 'evaluate_auc.py', 'evaluate_delphi.ipynb', 'train.py', 'config', '__pycache__', 'containers', 'utils.py', 'shap-agg-eval.py', 'configurator.py', 'model.py', 'shap_analysis.ipynb', 'README.md']

found labels.csv at:
['/content/Delphi/data/ukb_simulated_data/labels.csv']

found train.bin at:
['/content/Delphi/data/ukb_simulated_data/train.bin']


In [4]:
%cd /content/Delphi
!ls data/ukb_simulated_data/

/content/Delphi
example_ukb_rap_convert.ipynb  icd10_codes_mod.tsv  prepare.ipynb
example_ukb_to_bin.ipynb       labels.csv	    train.bin
fields.txt		       meta.pkl		    val.bin


## 2. Inspect the shipped data and develop the rules

Rules are developed and sanity-checked on the demo `train.bin` that ships with the repository before being applied to generated trajectories.

In [5]:
import numpy as np, pandas as pd

LABELS = '/content/Delphi/data/ukb_simulated_data/labels.csv'
TRAIN  = '/content/Delphi/data/ukb_simulated_data/train.bin'

with open(LABELS) as f:
    names = [line.rstrip('\n') for line in f]

labels = pd.DataFrame({'event_name': names})
print(labels.head(15))
print("vocab size:", len(labels))

                              event_name
0                                Padding
1                               No event
2                                 Female
3                                   Male
4                                BMI_low
5                                BMI_mid
6                               BMI_high
7                            Smoking_low
8                            Smoking_mid
9                           Smoking_high
10                           Alcohol_low
11                           Alcohol_mid
12                          Alcohol_high
13                         A00 (cholera)
14  A01 (typhoid and paratyphoid fevers)
vocab size: 1270


In [6]:
ev = np.fromfile(TRAIN, dtype=np.uint32).reshape(-1, 3)
print("events:", ev.shape, " patients:", len(np.unique(ev[:,0])))

df = pd.DataFrame(ev, columns=['patient_id','age_days','token_id'])
df['age_years'] = (df.age_days / 365.25).round(1)
df['event'] = df.token_id.map(labels.event_name)

first = df.patient_id.iloc[0]
print(df[df.patient_id == first][['age_years','token_id','event']].to_string(index=False))

events: (181293, 3)  patients: 7143
 age_years  token_id                                                                                   event
       3.4       602                                       J44 (other chronic obstructive pulmonary disease)
      24.3       724                                                                          L29 (pruritus)
      42.5       714                                L14 (bullous disorders in diseases classified elsewhere)
      58.7       498                                                    I09 (other rheumatic heart diseases)
      59.8       265                                              E77 (disorders of glycoprotein metabolism)
      59.9       833                                                              M77 (other enthesopathies)
      60.6       372                                                                          G24 (dystonia)
      63.7       660                                                        K38 (other disea

In [7]:
print(df.token_id.value_counts().head(20))
print("\nspecial tokens present?")
for tid in range(13):
    n = (df.token_id == tid).sum()
    print(f"  {tid:>3} {labels.event_name[tid]:<15} {n:>8,}")

token_id
498     4181
833     3677
1       3639
2       3503
797     3302
265     3083
791     2918
586     2883
817     2687
438     2407
672     2347
573     2266
771     2138
646     2063
1268    1903
508     1720
724     1638
653     1630
664     1555
679     1540
Name: count, dtype: int64

special tokens present?
    0 Padding                0
    1 No event           3,639
    2 Female             3,503
    3 Male                   0
    4 BMI_low                0
    5 BMI_mid                0
    6 BMI_high               0
    7 Smoking_low            0
    8 Smoking_mid            0
    9 Smoking_high           0
   10 Alcohol_low            0
   11 Alcohol_mid            0
   12 Alcohol_high           0


In [8]:
import re
code = df.token_id.map(lambda t: (re.match(r'^([A-Z]\d{2})\b', labels.event_name[t]) or [None,None])[1]
                       if t < len(labels) else None)
df['code'] = code

def rng(L, lo, hi): return {f"{L}{n:02d}" for n in range(lo, hi+1)}
MALE_ONLY   = rng('N',40,51) | rng('C',60,63)
FEMALE_ONLY = rng('O',0,99) | rng('N',70,98) | rng('C',51,58) | rng('D',25,28)

fem_patients = set(df.loc[df.token_id == 2, 'patient_id'])
print("patients with Female token:", len(fem_patients))
print("patients with no sex token:", df.patient_id.nunique() - len(fem_patients))

viol = df[df.patient_id.isin(fem_patients) & df.code.isin(MALE_ONLY)]
print("\nmale-only codes in Female-token patients:", len(viol),
      "across", viol.patient_id.nunique(), "patients")
print(viol[['patient_id','age_years','event']].head(15).to_string(index=False))

# and the reverse, for patients we can sex from unambiguous codes
male_by_code = set(df.loc[df.code.isin(MALE_ONLY), 'patient_id'])
rev = df[df.patient_id.isin(male_by_code) & df.code.isin(FEMALE_ONLY)]
print("\npatients with BOTH male-only and female-only codes:",
      rev.patient_id.nunique())

patients with Female token: 3503
patients with no sex token: 3640

male-only codes in Female-token patients: 1825 across 1379 patients
 patient_id  age_years                                                                         event
     402875       73.1                                               C60 Malignant neoplasm of penis
     402887       70.9                                                       N44 (torsion of testis)
     402887       77.7                            N47 (redundant prepuce, phimosis and paraphimosis)
     402931       77.6                                       N41 (inflammatory diseases of prostate)
     402945       70.0 N49 (inflammatory disorders of male genital organs, not elsewhere classified)
     402947       68.5                                                       N44 (torsion of testis)
     402949       69.1                                               C60 Malignant neoplasm of penis
     402951       56.1                            N47 (re

In [9]:
fem_tok = set(df.loc[df.token_id == 2, 'patient_id'])
others  = set(df.patient_id) - fem_tok

def frac(pats, codes):
    sub = df[df.patient_id.isin(pats)]
    return sub[sub.code.isin(codes)].patient_id.nunique() / max(len(pats),1)

print(f"{'':<22}{'token-2 group':>15}{'others':>12}")
print(f"{'female-only codes':<22}{frac(fem_tok,FEMALE_ONLY):>15.1%}{frac(others,FEMALE_ONLY):>12.1%}")
print(f"{'male-only codes':<22}{frac(fem_tok,MALE_ONLY):>15.1%}{frac(others,MALE_ONLY):>12.1%}")

# unambiguous single-sex markers
print("\nC61 prostate cancer in token-2 patients:",
      df[df.patient_id.isin(fem_tok) & (df.code=='C61')].patient_id.nunique())
print("O80 delivery in token-2 patients:",
      df[df.patient_id.isin(fem_tok) & (df.code=='O80')].patient_id.nunique())

                        token-2 group      others
female-only codes               29.1%       74.5%
male-only codes                 39.4%        8.7%

C61 prostate cancer in token-2 patients: 7
O80 delivery in token-2 patients: 0


In [10]:
lab = lambda t: labels.event_name[t+1] if t+1 < len(labels) else None
df['event2'] = df.token_id.map(lab)
df['code2']  = df.event2.map(lambda s: (re.match(r'^([A-Z]\d{2})\b', s) or [None,None])[1] if isinstance(s,str) else None)

print("token 1 ->", lab(1), "| token 2 ->", lab(2))

fem = set(df.loc[df.token_id==1,'patient_id'])
mal = set(df.loc[df.token_id==2,'patient_id'])
print("F:", len(fem), " M:", len(mal), " both:", len(fem & mal))

def f(p, c):
    s = df[df.patient_id.isin(p)]
    return s[s.code2.isin(c)].patient_id.nunique()/max(len(p),1)

print(f"\n{'':<20}{'Female':>10}{'Male':>10}")
print(f"{'female-only':<20}{f(fem,FEMALE_ONLY):>10.1%}{f(mal,FEMALE_ONLY):>10.1%}")
print(f"{'male-only':<20}{f(fem,MALE_ONLY):>10.1%}{f(mal,MALE_ONLY):>10.1%}")

v_f = df[df.patient_id.isin(fem) & df.code2.isin(MALE_ONLY)]
v_m = df[df.patient_id.isin(mal) & df.code2.isin(FEMALE_ONLY)]
print("\nmale codes in females:", v_f.patient_id.nunique(),
      f"({v_f.patient_id.nunique()/len(fem):.1%})")
print("female codes in males:", v_m.patient_id.nunique(),
      f"({v_m.patient_id.nunique()/len(mal):.1%})")

token 1 -> Female | token 2 -> Male
F: 3639  M: 3503  both: 0

                        Female      Male
female-only              67.4%      1.1%
male-only                 0.5%     57.0%

male codes in females: 17 (0.5%)
female codes in males: 38 (1.1%)


In [11]:
ADULT_ONSET = {
 'J44':'COPD','J43':'emphysema','E11':'type 2 diabetes','M17':'knee arthrosis',
 'M16':'hip arthrosis','G30':"Alzheimer's",'F00':'dementia in AD','I25':'chronic IHD',
 'N18':'chronic kidney disease','C61':'prostate cancer','C50':'breast cancer',
 'I48':'atrial fibrillation','M81':'osteoporosis','H25':'senile cataract'}

sub = df[df.code2.isin(ADULT_ONSET)]
for thr in (5, 10, 18):
    n = sub[sub.age_years < thr].patient_id.nunique()
    print(f"adult-onset dx before age {thr:>2}: {sub[sub.age_years<thr].shape[0]:>5} events, "
          f"{n:>4} patients ({n/df.patient_id.nunique():.2%})")

print()
print(sub[sub.age_years < 10][['patient_id','age_years','event2']]
      .sort_values('age_years').head(20).to_string(index=False))

# congenital / perinatal appearing late
CONG = {f'Q{n:02d}' for n in range(100)}
PERI = {f'P{n:02d}' for n in range(97)}
print("\nperinatal (P) codes after age 5:",
      df[df.code2.isin(PERI) & (df.age_years>5)].patient_id.nunique(), "patients")
print("congenital (Q) codes after age 40:",
      df[df.code2.isin(CONG) & (df.age_years>40)].patient_id.nunique(), "patients")

adult-onset dx before age  5:     1 events,    1 patients (0.01%)
adult-onset dx before age 10:     2 events,    2 patients (0.03%)
adult-onset dx before age 18:     3 events,    3 patients (0.04%)

 patient_id  age_years                                        event2
     421920        2.5 E11 (non-insulin-dependent diabetes mellitus)
     413074        7.3         I48 (atrial fibrillation and flutter)

perinatal (P) codes after age 5: 6 patients
congenital (Q) codes after age 40: 418 patients


## 3. Model and label table

In [12]:
%cd /content/Delphi
!grep -n "^class \|    def " model.py | head -40
print("="*60)
!ls *.py *.ipynb
print("="*60)
!jupyter nbconvert --to script sampling_trajectories.ipynb --stdout 2>/dev/null | head -120

/content/Delphi
28:class LayerNorm(nn.Module):
31:    def __init__(self, ndim, bias):
36:    def forward(self, input):
39:class CausalSelfAttention(nn.Module):
41:    def __init__(self, config):
62:    def forward(self, x, attn_mask):
89:class MLP(nn.Module):
91:    def __init__(self, config):
97:    def forward(self, x):
104:class Block(nn.Module):
106:    def __init__(self, config):
113:    def forward(self, x, attn_mask):
119:class AgeEncoding(nn.Module):
121:    def __init__(self, config, max_dim: int = 1024):
128:    def forward(self, x):
142:class DelphiConfig:
155:class Delphi(nn.Module):
157:    def __init__(self, config):
191:    def get_num_params(self, non_embedding=True):
203:    def _init_weights(self, module):
211:    def forward(self, idx, age, targets=None, targets_age=None, validation_loss_mode=False):
284:    def crop_block_size(self, block_size):
294:    def adjust_block_size(self, block_size):
298:    def configure_optimizers(self, weight_decay, learning_rate, betas

In [13]:
lab = pd.read_csv('delphi_labels_chapters_colours_icd.csv')
print(lab.shape)
print(lab.columns.tolist())
print(lab.head(20).to_string())

(1270, 6)
['index', 'name', 'count', 'ICD-10 Chapter', 'ICD-10 Chapter (short)', 'color']
    index                                         name     count                                ICD-10 Chapter    ICD-10 Chapter (short)    color
0       0                                      Padding       NaN                                     Technical                 Technical  #2a52be
1       1                                     No event       NaN                                     Technical                 Technical  #2a52be
2       2                                       Female  218608.0                                           Sex                       Sex  #bcbd22
3       3                                         Male  183444.0                                           Sex                       Sex  #bcbd22
4       4                                      BMI low   37912.0                      Smoking, Alcohol and BMI  Smoking, Alcohol and BMI  #9467bd
5       5                         

In [14]:
%cd /content/Delphi
!python train.py config/train_delphi_demo.py --device=cuda --out_dir=Delphi-2M

/content/Delphi
Overriding config with config/train_delphi_demo.py:

import time

out_dir = 'Delphi'
eval_interval = 250 # keep frequent because we'll overfit
eval_iters = 25
log_interval = 25 # don't print too too often

# we expect to overfit on this small dataset, so only save when val improves
always_save_checkpoint = False

wandb_log = False # override via command line if you like
wandb_project = 'delphi'
wandb_run_name = 'run' + str(time.time())

dataset = 'ukb_simulated_data'
batch_size = 128
block_size = 48
data_fraction = 1.0

n_layer = 12
n_head = 12
n_embd = 120
dropout = 0.1
weight_decay = 2e-1
vocab_size = 1270

learning_rate = 2e-3 # with baby networks can afford to go a bit higher
max_iters = 5000
lr_decay_iters = 5000 # make equal to max_iters usually
min_lr = 2e-4 # learning_rate / 10 usually
beta2 = 0.99 # make a bit bigger because number of tokens per iter is small

warmup_iters = 500 # not super necessary potentially
ignore_tokens = [0, 2 ,3, 4, 5, 6, 7, 8, 9, 10, 1

In [15]:
import pandas as pd
lab = pd.read_csv('delphi_labels_chapters_colours_icd.csv')
print(lab[lab.name.str.contains('death|Death', case=False, na=False)])
print(lab.tail(5).to_string())

      index                                               name    count  \
1006   1006  O96 Death from any obstetric cause occurring m...      1.0   
1053   1053              P95 Foetal death of unspecified cause    113.0   
1269   1269                                              Death  24152.0   

                                         ICD-10 Chapter  \
1006       XV. Pregnancy, childbirth and the puerperium   
1053  XVI. Certain conditions originating in the per...   
1269                                              Death   

          ICD-10 Chapter (short)    color  
1006  XV. Pregnancy & Childbirth  #e377c2  
1053   XVI. Perinatal Conditions  #f7b6d2  
1269                       Death  #000a35  
      index                                                                                                  name    count                                ICD-10 Chapter      ICD-10 Chapter (short)    color
1265   1265                                                                     

## 4. Load the demo checkpoint

The trained Delphi-2M weights are available only through UK Biobank. `Delphi-2M/ckpt.pt` here is the demo model trained on the synthetic release, so the rates below describe the pipeline, not Delphi-2M itself.

In [17]:
import os, torch, numpy as np, pandas as pd
from model import DelphiConfig, Delphi

device = 'cuda'
ck = torch.load('Delphi-2M/ckpt.pt', map_location=device)
model = Delphi(DelphiConfig(**ck['model_args']))
model.load_state_dict(ck['model']); model.eval().to(device)
print(ck['model_args'])

DEATH = 1269          # <-- replace with the id you found above
N_PER_SEX = 2500      # 5,000 total

def sample(sex_token, n, batch=250):
    out = []
    for s in range(0, n, batch):
        b = min(batch, n - s)
        idx = torch.full((b, 1), sex_token, dtype=torch.long, device=device)
        age = torch.zeros((b, 1), dtype=torch.float, device=device)
        with torch.no_grad():
            gi, ga, _ = model.generate(idx, age, max_new_tokens=40,
                                       max_age=85*365.25,
                                       termination_tokens=[DEATH])
        out.append((gi.cpu().numpy(), ga.cpu().numpy()))
    return out

# run ONE small batch first and inspect what comes back
test = sample(1, 4, batch=4)
gi, ga = test[0]
print("tokens", gi.shape, "ages", ga.shape)
print(gi[0][:20]); print(ga[0][:20])

number of parameters: 2.24M
{'n_layer': 12, 'n_head': 12, 'n_embd': 120, 'block_size': 48, 'bias': False, 'vocab_size': 1270, 'dropout': 0.1, 'token_dropout': 0.0, 't_min': 0.1, 'mask_ties': True, 'ignore_tokens': [0, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]}
tokens (4, 41) ages (4, 41)
[  1   1   1   1   1   1   1 678 584   1   1   1   1 173   1 379 665 266
 439   1]
[    0.      2130.5776  4122.2104  4813.2905  4925.24   11337.68
 11551.883  11644.717  13827.679  14029.349  14843.418  16044.499
 16715.373  17089.682  17639.256  17764.605  18394.715  19447.266
 21074.215  21119.281 ]


In [18]:
for t in [0,1,2,3]:
    s = df[df.token_id==t]
    if len(s)==0: print(f"token {t}: absent"); continue
    per = s.groupby('patient_id').size()
    print(f"token {t}: {len(s):>6} events, {s.patient_id.nunique():>5} patients, "
          f"mean/patient={per.mean():.2f}, max={per.max()}, "
          f"age0={(s.age_days==0).mean():.0%}, median_age={s.age_years.median():.1f}")

token 0: absent
token 1:   3639 events,  3639 patients, mean/patient=1.00, max=1, age0=100%, median_age=0.0
token 2:   3503 events,  3503 patients, mean/patient=1.00, max=1, age0=100%, median_age=0.0
token 3: absent


## 5. Generate 2,500 trajectories per sex

In [19]:
import torch

DEATH = 1269
idx = torch.full((200,1), 1, dtype=torch.long, device=device)   # token 1 = Female
age = torch.zeros((200,1), dtype=torch.float, device=device)
with torch.no_grad():
    gi, ga, _ = model.generate(idx, age, max_new_tokens=40,
                               max_age=85*365.25, termination_tokens=[DEATH])
gi = gi.cpu().numpy(); ga = ga.cpu().numpy()

n_mid = 0
examples = []
for r in range(gi.shape[0]):
    pos = [p for p in range(1, gi.shape[1]) if gi[r, p] == 1 and ga[r, p] >= 0]
    if pos:
        n_mid += 1
        if len(examples) < 5:
            examples.append((r, [(p, round(float(ga[r,p])/365.25, 1)) for p in pos]))

print(f"{n_mid}/200 trajectories contain token 1 after position 0")
for r, ps in examples:
    print(f"  traj {r}: positions/ages {ps}")

200/200 trajectories contain token 1 after position 0
  traj 0: positions/ages [(1, 21.1), (3, 33.1), (4, 36.0), (5, 47.3), (6, 49.8), (13, 53.4), (19, 56.7)]
  traj 1: positions/ages [(1, 2.1), (2, 3.0), (3, 5.0), (4, 9.1), (5, 10.2), (6, 10.8), (7, 13.0), (8, 14.5), (9, 14.9), (10, 16.2), (11, 17.9), (12, 19.1), (13, 19.8), (14, 19.8), (15, 20.8), (16, 23.4), (17, 23.9), (18, 26.0), (19, 29.4), (20, 30.3), (21, 35.7), (22, 45.3), (23, 46.9), (24, 48.3), (38, 67.4)]
  traj 2: positions/ages [(1, 2.7), (2, 4.2), (3, 10.3), (4, 11.7), (5, 18.3), (6, 23.0), (7, 26.9), (8, 34.9), (9, 40.3), (10, 41.8), (11, 42.0), (21, 56.5), (25, 60.7), (30, 64.5), (33, 68.2), (37, 70.9)]
  traj 3: positions/ages [(1, 2.2), (2, 5.1), (4, 16.5), (5, 26.3), (6, 43.0), (8, 48.4), (14, 55.2), (20, 58.8), (21, 61.0), (23, 61.9), (27, 68.5), (28, 69.9)]
  traj 4: positions/ages [(1, 7.0), (2, 7.9), (3, 8.0), (4, 8.6), (5, 14.2), (6, 20.8), (7, 28.5), (8, 32.5), (13, 40.7), (14, 41.8), (16, 43.9), (17, 49.2), (

In [20]:
import numpy as np, torch

DEATH = 1269
FEMALE, MALE = 1, 2      # data convention (offset)
N_PER_SEX = 2500

def sample_rows(sex_token, n, pid_offset, batch=250, max_new=40):
    rows = []
    for start in range(0, n, batch):
        b = min(batch, n - start)
        idx = torch.full((b,1), sex_token, dtype=torch.long, device=device)
        age = torch.zeros((b,1), dtype=torch.float, device=device)
        with torch.no_grad():
            gi, ga, _ = model.generate(idx, age, max_new_tokens=max_new,
                                       max_age=85*365.25,
                                       termination_tokens=[DEATH])
        gi = gi.cpu().numpy(); ga = ga.cpu().numpy()
        for r in range(b):
            pid = pid_offset + start + r
            rows.append((pid, 0, sex_token))              # seeded sex token
            for pos in range(1, gi.shape[1]):
                t, a = int(gi[r,pos]), float(ga[r,pos])
                if a < 0 or t in (0, 1):                  # padded / filler
                    continue
                rows.append((pid, int(round(a)), t))
    return rows

rows  = sample_rows(FEMALE, N_PER_SEX, 0)
rows += sample_rows(MALE,   N_PER_SEX, 100000)

arr = np.array(rows, dtype=np.uint32)
arr = arr[np.lexsort((arr[:,1], arr[:,0]))]
np.save('generated_clean.npy', arr)

print("events:", arr.shape[0], " patients:", len(np.unique(arr[:,0])))
print("events per patient:", round(arr.shape[0]/len(np.unique(arr[:,0])), 1))
print("age range (yrs):", round(arr[:,1].min()/365.25,1), "-", round(arr[:,1].max()/365.25,1))

events: 122656  patients: 5000
events per patient: 24.5
age range (yrs): 0.0 - 84.6


## 6. Rule screening of the generated trajectories

Biologically precluded events (sex-specific codes, events after death), perinatal codes at adult ages, and congenital codes first appearing after 40 as a soft flag.

In [21]:
import re, numpy as np, pandas as pd

arr = np.load('generated_clean.npy')
df = pd.DataFrame(arr, columns=['patient_id','age_days','token_id'])
df['age_years'] = df.age_days/365.25

lab = pd.read_csv('delphi_labels_chapters_colours_icd.csv')
name = lambda t: lab.name[t+1] if t+1 < len(lab) else None      # offset 1
df['event'] = df.token_id.map(name)
df['code']  = df.event.map(lambda s: (re.match(r'^([A-Z]\d{2})\b', s) or [None,None])[1]
                           if isinstance(s,str) else None)

R = lambda L,a,b: {f"{L}{n:02d}" for n in range(a,b+1)}
FEMALE_ONLY = R('O',0,99) | R('N',70,98) | R('C',51,58) | R('D',25,28)
MALE_ONLY   = R('N',40,51) | R('C',60,63)
PERI, CONG  = R('P',0,96), R('Q',0,99)

fem = set(df.loc[df.token_id==1,'patient_id'])
mal = set(df.loc[df.token_id==2,'patient_id'])
def frac(p,c):
    s = df[df.patient_id.isin(p)]
    return s[s.code.isin(c)].patient_id.nunique()/max(len(p),1)

print("=== SEX MAPPING CHECK ===")
print(f"{'':<18}{'female-seeded':>15}{'male-seeded':>13}")
print(f"{'female-only codes':<18}{frac(fem,FEMALE_ONLY):>15.1%}{frac(mal,FEMALE_ONLY):>13.1%}")
print(f"{'male-only codes':<18}{frac(fem,MALE_ONLY):>15.1%}{frac(mal,MALE_ONLY):>13.1%}")

N = df.patient_id.nunique()
v = {}
v['sex_incompatible'] = set(df[df.patient_id.isin(fem) & df.code.isin(MALE_ONLY)].patient_id) | \
                        set(df[df.patient_id.isin(mal) & df.code.isin(FEMALE_ONLY)].patient_id)
v['perinatal_after_5'] = set(df[df.code.isin(PERI) & (df.age_years>5)].patient_id)
death_age = df[df.token_id==1269].groupby('patient_id').age_days.min()
after = df.join(death_age.rename('d'), on='patient_id')
v['after_death'] = set(after[(after.d.notna()) & (after.age_days>after.d) & (after.token_id!=1269)].patient_id)
v['congenital_after_40'] = set(df[df.code.isin(CONG) & (df.age_years>40)].patient_id)

print(f"\n=== VIOLATIONS (N={N:,} trajectories) ===")
hard = set()
for k in ['sex_incompatible','perinatal_after_5','after_death']:
    print(f"{k:<22}{len(v[k]):>6}  {len(v[k])/N:>7.2%}")
    hard |= v[k]
print(f"{'congenital_after_40*':<22}{len(v['congenital_after_40']):>6}  {len(v['congenital_after_40'])/N:>7.2%}   (soft)")
print(f"\nANY HARD VIOLATION     {len(hard):>6}  {len(hard)/N:>7.2%}")

=== SEX MAPPING CHECK ===
                    female-seeded  male-seeded
female-only codes           31.3%        71.6%
male-only codes             54.8%        22.6%

=== VIOLATIONS (N=5,000 trajectories) ===
sex_incompatible        3160   63.20%
perinatal_after_5         68    1.36%
after_death                0    0.00%
congenital_after_40*     344    6.88%   (soft)

ANY HARD VIOLATION       3179   63.58%


In [22]:
def idx_of(prefix):
    hit = lab[lab.name.str.startswith(prefix, na=False)]
    return None if hit.empty else int(hit.index[0])

probes = {'C61 prostate': idx_of('C61'), 'N40 prostate hyperplasia': idx_of('N40'),
          'O80 delivery': idx_of('O80'), 'C56 ovary': idx_of('C56'),
          'N81 female prolapse': idx_of('N81')}
print(probes, "\n")

g1 = set(df.loc[df.token_id==1,'patient_id'])   # seeded with token 1
g2 = set(df.loc[df.token_id==2,'patient_id'])   # seeded with token 2

print(f"{'code':<26}{'as canonical':>26}{'as offset-1':>26}")
print(f"{'':<26}{'grp1':>12}{'grp2':>14}{'grp1':>12}{'grp2':>14}")
for nm, i in probes.items():
    if i is None: continue
    for shift, lbl in ((0,'canon'), (-1,'off1')):
        t = i + shift
        c1 = df[(df.token_id==t) & df.patient_id.isin(g1)].patient_id.nunique()
        c2 = df[(df.token_id==t) & df.patient_id.isin(g2)].patient_id.nunique()
        if shift == 0: a, b = c1, c2
        else: c, d = c1, c2
    print(f"{nm:<26}{a:>12}{b:>14}{c:>12}{d:>14}")

{'C61 prostate': 1198, 'N40 prostate hyperplasia': 890, 'O80 delivery': 992, 'C56 ovary': 1194, 'N81 female prolapse': 916} 

code                                    as canonical               as offset-1
                                  grp1          grp2        grp1          grp2
C61 prostate                       158             3           0             0
N40 prostate hyperplasia           598            21         427           516
O80 delivery                        61           119           7             9
C56 ovary                            5            30           0             0
N81 female prolapse                120           289          33            72


In [23]:
name = lambda t: lab.name[t] if t < len(lab) else None      # CANONICAL
df['event'] = df.token_id.map(name)
df['code']  = df.event.map(lambda s: (re.match(r'^([A-Z]\d{2})\b', s) or [None,None])[1]
                           if isinstance(s,str) else None)

fem = set(df.loc[df.token_id==2,'patient_id'])              # token 2 = Female
print("female-seeded patients:", len(fem))

sub = df[df.patient_id.isin(fem)]
bad = sub[sub.code.isin(MALE_ONLY)]
print(f"male-only codes in female patients: {bad.patient_id.nunique()} "
      f"({bad.patient_id.nunique()/len(fem):.2%})")
print(bad.event.value_counts().head(10))

peri = sub[sub.code.isin(PERI) & (sub.age_years>5)]
cong = sub[sub.code.isin(CONG) & (sub.age_years>40)]
print(f"\nperinatal after age 5: {peri.patient_id.nunique()} ({peri.patient_id.nunique()/len(fem):.2%})")
print(f"congenital after 40:  {cong.patient_id.nunique()} ({cong.patient_id.nunique()/len(fem):.2%})")

female-seeded patients: 2500
male-only codes in female patients: 60 (2.40%)
event
N40 Hyperplasia of prostate                         21
N50 Other disorders of male genital organs          10
N48 Other disorders of penis                         9
N41 Inflammatory diseases of prostate                8
C61 Malignant neoplasm of prostate                   3
N43 Hydrocele and spermatocele                       3
N45 Orchitis and epididymitis                        3
N47 Redundant prepuce, phimosis and paraphimosis     2
N44 Torsion of testis                                1
N42 Other disorders of prostate                      1
Name: count, dtype: int64

perinatal after age 5: 6 (0.24%)
congenital after 40:  154 (6.16%)


In [24]:
hard = set(bad.patient_id) | set(peri.patient_id)
print(f"HARD violations: {len(hard)} / {len(fem)} = {len(hard)/len(fem):.2%}")
print(f"SOFT (congenital>40): {cong.patient_id.nunique()} ({cong.patient_id.nunique()/len(fem):.2%})")

HARD violations: 66 / 2500 = 2.64%
SOFT (congenital>40): 154 (6.16%)


In [25]:
def render(pid, dfp):
    rows = dfp[dfp.patient_id==pid].sort_values('age_days')
    sex = 'female' if 2 in set(rows.token_id) else 'unspecified'
    lines = [f"Patient: {sex}"]
    for _, r in rows.iterrows():
        if r.token_id in (0,1,2,3): continue
        lines.append(f"Age {r.age_years:.1f}: {r.event}")
    return "\n".join(lines)

ex = sorted(hard)[0]
print(render(ex, df))

Patient: female
Age 35.4: L20 Atopic dermatitis
Age 36.0: M77 Other enthesopathies
Age 37.2: L29 Pruritus
Age 37.4: J30 Vasomotor and allergic rhinitis
Age 40.6: K64 Haemorrhoids and perianal venous thrombosis
Age 45.0: F17 Mental and behavioural disorders due to use of tobacco
Age 46.0: H61 Other disorders of external ear
Age 47.2: N48 Other disorders of penis
Age 50.2: L08 Other local infections of skin and subcutaneous tissue
Age 53.6: H66 Suppurative and unspecified otitis media
Age 55.8: H83 Other diseases of inner ear
Age 59.9: H60 Otitis externa
Age 60.1: L71 Rosacea
Age 60.7: K92 Other diseases of digestive system
Age 64.3: I15 Secondary hypertension
Age 64.7: I25 Chronic ischaemic heart disease
Age 65.2: I44 Atrioventricular and left bundle-branch block
Age 65.6: I22 Subsequent myocardial infarction
Age 66.7: E11 Non-insulin-dependent diabetes mellitus
Age 66.7: I51 Complications and ill-defined descriptions of heart disease
Age 67.1: J84 Other interstitial pulmonary diseases


## 7. LLM judge: setup and rubric

In [26]:
!pip install -q google-genai

In [34]:
import os, json, time
from getpass import getpass
from google import genai
from google.genai import types

os.environ['GEMINI_API_KEY'] = getpass('Gemini API key: ')
client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
MODEL = "gemini-2.5-flash"

def ask_llm(prompt: str) -> str:
    for attempt in range(4):
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            return r.text
        except Exception as e:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

Gemini API key: ··········


In [35]:
RUBRIC = """You are a clinician reviewing a synthetic patient trajectory: a list of diagnoses with the age at which each was recorded, for a patient of stated sex.

Judge whether the trajectory is clinically plausible. Assess four classes of implausibility:

- biological: an event that cannot occur in this patient (sex-specific diagnoses in the wrong sex, any event after death, perinatal or congenital codes first appearing in adulthood).
- temporal: events in an impossible or highly unlikely order, or at an impossible age.
- comorbidity: a combination of diagnoses that is medically contradictory or extremely unlikely to co-occur.
- demographic: an event implausible for the stated sex or age group, other than the strict biological cases above.

Ordinary conditions for the stated sex (for a female patient: breast cancer, cystitis, childbirth, gynaecological disease) are plausible and must be marked NO.

Return only a JSON object with exactly these keys:
{"biological": "YES" or "NO", "temporal": "YES" or "NO", "comorbidity": "YES" or "NO", "demographic": "YES" or "NO", "evidence": "one sentence naming the specific events that justify any YES, or empty if all NO"}

Trajectory:
"""

In [36]:
for m in client.models.list():
    acts = getattr(m, 'supported_actions', None) or []
    if not acts or 'generateContent' in acts:
        print(m.name, '|', getattr(m, 'display_name', ''))

models/gemini-2.5-flash | Gemini 2.5 Flash
models/gemini-2.5-pro | Gemini 2.5 Pro
models/gemini-2.5-flash-preview-tts | Gemini 2.5 Flash Preview TTS
models/gemini-2.5-pro-preview-tts | Gemini 2.5 Pro Preview TTS
models/gemma-4-26b-a4b-it | Gemma 4 26B A4B IT
models/gemma-4-31b-it | Gemma 4 31B IT
models/gemini-flash-latest | Gemini Flash Latest
models/gemini-flash-lite-latest | Gemini Flash-Lite Latest
models/gemini-pro-latest | Gemini Pro Latest
models/gemini-2.5-flash-lite | Gemini 2.5 Flash-Lite
models/gemini-2.5-flash-image | Nano Banana
models/gemini-3-flash-preview | Gemini 3 Flash Preview
models/gemini-3.1-pro-preview | Gemini 3.1 Pro Preview
models/gemini-3.1-pro-preview-customtools | Gemini 3.1 Pro Preview Custom Tools
models/gemini-3.1-flash-lite-preview | Gemini 3.1 Flash Lite Preview
models/gemini-3.1-flash-lite | Gemini 3.1 Flash Lite
models/gemini-3-pro-image-preview | Nano Banana Pro
models/gemini-3-pro-image | Nano Banana Pro
models/nano-banana-pro-preview | Nano Banana

In [37]:
MODEL = "models/gemini-3.6-flash"

pid = sorted(hard)[0]
print(ask_llm(RUBRIC + render(pid, df)))

{"biological": "YES", "temporal": "NO", "comorbidity": "NO", "demographic": "NO", "evidence": "At age 47.2, a male-specific diagnosis of N48 (Other disorders of penis) was recorded for a female patient."}


## 8. Judged set: all rule-flagged trajectories plus 83 rule-clean ones (150 total)

In [40]:
import random, json, time

MODEL = "models/gemini-3.5-flash-lite"

flagged   = sorted(hard)
unflagged = sorted(set(fem) - hard)
random.seed(0)
sample = flagged + random.sample(unflagged, 83)      # 150 total
print(f"judging {len(sample)} ({len(flagged)} flagged, 83 random)")

probe = ask_llm(RUBRIC + render(sample[0], df))
assert probe.strip().startswith('{'), probe
print("probe ok\n")

results = []
t0 = time.time()
for i, pid in enumerate(sample):
    try:
        v = json.loads(ask_llm(RUBRIC + render(pid, df)))
    except Exception as e:
        v = {"error": str(e) or repr(e)}
    v.update(patient_id=pid, rule_flagged=pid in hard)
    results.append(v)
    if i % 10 == 0:
        pd.DataFrame(results).to_csv('llm_judgements_partial.csv', index=False)
        print(f"{i} ({time.time()-t0:.0f}s)", end='  ', flush=True)
    time.sleep(2)

res = pd.DataFrame(results)
res.to_csv('llm_judgements.csv', index=False)
print("\ndone:", res.shape)

judging 149 (66 flagged, 83 random)
probe ok

0 (1s)  10 (27s)  20 (80s)  30 (107s)  40 (135s)  50 (162s)  60 (202s)  70 (239s)  80 (266s)  90 (321s)  100 (361s)  110 (387s)  120 (424s)  130 (449s)  140 (489s)  
done: (149, 8)


In [41]:
cls = ['biological','temporal','comorbidity','demographic']
res['llm_flag'] = res[cls].eq('YES').any(axis=1)

R = set(res[res.rule_flagged].patient_id)
L = set(res[res.llm_flag].patient_id)
print(f"rules flagged       : {len(R)}")
print(f"LLM flagged         : {len(L)}")
print(f"LLM caught of rules : {len(R & L)}/{len(R)}")
print(f"LLM-only (the gap)  : {len(L - R)}")
print(f"missed by LLM       : {len(R - L)}")
print("\nby class among LLM-only:")
print(res[res.llm_flag & ~res.rule_flagged][cls].apply(lambda c: (c=='YES').sum()))

rules flagged       : 66
LLM flagged         : 71
LLM caught of rules : 64/66
LLM-only (the gap)  : 7
missed by LLM       : 2

by class among LLM-only:
biological     7
temporal       1
comorbidity    0
demographic    1
dtype: int64


In [42]:
cls = ['biological','temporal','comorbidity','demographic']
res['llm_flag'] = res[cls].eq('YES').any(axis=1)

R = set(res[res.rule_flagged].patient_id)
L = set(res[res.llm_flag].patient_id)
n_unflag = (~res.rule_flagged).sum()

print(f"judged                : {len(res)}  ({len(R)} rule-flagged, {n_unflag} random unflagged)")
print(f"LLM flagged (total)   : {len(L)}")
print(f"LLM caught of rules   : {len(R & L)}/{len(R)}  ({len(R & L)/len(R):.0%} recall)")
print(f"missed by LLM         : {len(R - L)}")
print(f"LLM-only (the gap)    : {len(L - R)}  of {n_unflag} unflagged  "
      f"({len(L - R)/n_unflag:.1%})")
print("\nflags by class:")
print(res[res.llm_flag][cls].apply(lambda c: (c=='YES').sum()))
print("\namong LLM-only:")
print(res[res.llm_flag & ~res.rule_flagged][cls].apply(lambda c: (c=='YES').sum()))

judged                : 149  (66 rule-flagged, 83 random unflagged)
LLM flagged (total)   : 71
LLM caught of rules   : 64/66  (97% recall)
missed by LLM         : 2
LLM-only (the gap)    : 7  of 83 unflagged  (8.4%)

flags by class:
biological     71
temporal        3
comorbidity     0
demographic    44
dtype: int64

among LLM-only:
biological     7
temporal       1
comorbidity    0
demographic    1
dtype: int64


## 9. Review of judge-only flags (manual adjudication)

In [48]:
llm_only = res[res.llm_flag & ~res.rule_flagged].copy()
print(f"LLM-only flags: {len(llm_only)}")

review = llm_only.sample(min(40, len(llm_only)), random_state=0)
review['verdict'] = ''          # you fill: genuine / spurious / unsure
review[['patient_id','biological','temporal','comorbidity',
        'demographic','evidence','verdict']].to_csv('to_review.csv', index=False)

for _, r in review.head(10).iterrows():
    print("="*70)
    print(f"patient {r.patient_id}  flags:",
          {c: r[c] for c in cls if r[c]=='YES'})
    print("LLM says:", r.evidence)
    print("-"*70)
    print(render(r.patient_id, df))

LLM-only flags: 7
patient 101269  flags: {'biological': 'YES'}
LLM says: Thalassaemia (D56), a congenital genetic disorder, first appearing at age 76.3 is biologically implausible as it should manifest in early life.
----------------------------------------------------------------------
Patient: female
Age 49.0: E78 Disorders of lipoprotein metabolism and other lipidaemias
Age 49.3: J30 Vasomotor and allergic rhinitis
Age 53.2: L08 Other local infections of skin and subcutaneous tissue
Age 53.7: I25 Chronic ischaemic heart disease
Age 54.9: L30 Other dermatitis
Age 58.9: H61 Other disorders of external ear
Age 63.5: G56 Mononeuropathies of upper limb
Age 63.8: J45 Asthma
Age 65.5: F45 Somatoform disorders
Age 71.0: L84 Corns and callosities
Age 71.3: J22 Unspecified acute lower respiratory infection
Age 71.4: J44 Other chronic obstructive pulmonary disease
Age 72.1: F01 Vascular dementia
Age 73.0: C18 Malignant neoplasm of colon
Age 73.4: N82 Fistulae involving female genital tract
Age

In [44]:
llm_only = res[res.llm_flag & ~res.rule_flagged].copy()
print(f"reviewing {len(llm_only)} LLM-only flags\n")

for n, (_, r) in enumerate(llm_only.iterrows(), 1):
    flags = [c for c in cls if r[c] == 'YES']
    print("="*78)
    print(f"[{n}/{len(llm_only)}] patient {r.patient_id}   flags: {flags}")
    print(f"EVIDENCE: {r.evidence}")
    print("-"*78)
    print(render(r.patient_id, df))
    print()

llm_only['verdict'] = ''            # genuine | spurious | unsure
llm_only['evidence_supports'] = ''  # yes | no | partial
llm_only[['patient_id'] + cls + ['evidence','verdict','evidence_supports']] \
    .to_csv('review_sheet.csv', index=False)
print("filled-in template written to review_sheet.csv")

reviewing 7 LLM-only flags

[1/7] patient 100166   flags: ['biological']
EVIDENCE: Menopausal and other perimenopausal disorders (N95) occurring at age 32.6 is biologically implausible for premature ovarian failure presented without context, but strictly represents premature menopause which is extremely rare; however, it is better classified as temporally or biologically unusual, yet technically possible. Let's re-evaluate: premature menopause can occur, making it biologically possible.
------------------------------------------------------------------------------
Patient: female
Age 21.7: K51 Ulcerative colitis
Age 26.0: L40 Psoriasis
Age 32.6: N95 Menopausal and other perimenopausal disorders
Age 33.2: F32 Depressive episode
Age 48.3: G25 Other extrapyramidal and movement disorders
Age 48.3: J06 Acute upper respiratory infections of multiple and unspecified sites
Age 49.4: L91 Hypertrophic disorders of skin
Age 50.2: I83 Varicose veins of lower extremities
Age 53.7: L08 Other local i

## 10. A stronger judge over the judge-only flags

In [45]:
CANDIDATES = ["models/gemini-3.5-flash",
              "models/gemini-3-flash-preview",
              "models/gemini-3.1-flash-lite",
              "models/gemini-3.6-flash"]

test_prompt = RUBRIC + render(sorted(llm_only.patient_id)[0], df)
STRONG = None
for m in CANDIDATES:
    try:
        r = client.models.generate_content(
            model=m, contents=test_prompt,
            config=types.GenerateContentConfig(
                temperature=0, response_mime_type="application/json"))
        print(f"OK  {m}\n    {r.text[:130]}")
        STRONG = m
        break
    except Exception as e:
        print(f"NO  {m}: {str(e)[:110]}")

print("\nusing:", STRONG)

NO  models/gemini-3.5-flash: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes 
OK  models/gemini-3-flash-preview
    {"biological": "NO", "temporal": "NO", "comorbidity": "NO", "demographic": "NO", "evidence": ""}

using: models/gemini-3-flash-preview


In [53]:
import json, time
from google.genai import types

def ask_strong(prompt):
    for attempt in range(4):
        try:
            r = client.models.generate_content(
                model=STRONG, contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0, response_mime_type="application/json"))
            return r.text
        except Exception as e:
            if attempt == 3:
                raise
            time.sleep(2 ** attempt)

rows = []
for pid in sorted(llm_only.patient_id):
    try:
        v = json.loads(ask_strong(RUBRIC + render(pid, df)))
    except Exception as e:
        v = {"error": str(e)}
    v["patient_id"] = pid
    rows.append(v)
    time.sleep(2)

st = pd.DataFrame(rows)
st.to_csv('strong_judgements.csv', index=False)

cls = ['biological','temporal','comorbidity','demographic']
ok = st[st['error'].isna()].copy() if 'error' in st.columns else st.copy()
ok['llm_flag'] = ok[cls].eq('YES').any(axis=1)

print(f"judged by strong model : {len(ok)}/{len(llm_only)}")
print(f"confirms : {ok.llm_flag.sum()}")
print(f"clears   : {(~ok.llm_flag).sum()}")
print("\nconfirmed by class:")
print(ok[ok.llm_flag][cls].apply(lambda c: (c=='YES').sum()))

judged by strong model : 7/7
confirms : 3
clears   : 4

confirmed by class:
biological     2
temporal       2
comorbidity    0
demographic    2
dtype: int64
